In [1]:
import pandas as pd
import numpy as np
import torch
from transformer_time_series_enc_dec import train_model,InformerForecaster,create_dataloaders,TrainConfig,inverse_transform,init_weights
import plotly.express as px
import matplotlib.pyplot as plt
from metrics_leadlag import lead_lag_grid, lead_lag_report, plot_lead_lag
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
import json
from datetime import datetime
import os

def create_loss_plot(train_hist, val_hist, steps_hist, total_params=None):
    """Create a loss plot from training history"""
    df_train = pd.DataFrame({
        "Step": steps_hist,
        "Loss": train_hist,
        "Type": "Train"
    })
    if val_hist:
        val_steps, val_losses = zip(*val_hist)
        df_val = pd.DataFrame({
            "Step": val_steps,
            "Loss": val_losses,
            "Type": "Validation"
        })
        df_loss = pd.concat([df_train, df_val], ignore_index=True)
    else:
        df_loss = df_train
        
    title = "Training and Validation Loss"
    if total_params is not None:
        title += f"\nTotal Parameters: {total_params:,}"
    
    fig = px.line(df_loss, x="Step", y="Loss", color="Type", title=title)
    return fig, df_loss

def create_prediction_plots(model, val_loader, asset_idx, scaler, config, num_batches=10, save_dir=None):
    """Create prediction plots and calculate metrics for validation batches
    
    Args:
        model: The trained model
        val_loader: Validation data loader
        asset_idx: Index of the asset to predict
        scaler: Scaler used for data normalization
        config: Model configuration
        num_batches: Number of batches to visualize
        save_dir: If provided, save plots to this directory
        
    Returns:
        list: List of dictionaries containing metrics for each batch
    """
    metrics = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_batches:
                break
                
            # Unpack the batch tuple correctly
            x, timestamps = batch
            x = x.to(device)
            timestamps = timestamps.to(device)
            y_pred = model(x, timestamps)
            y_true = x[:, -model.pred_len:, asset_idx]

            # Move to CPU and convert to numpy
            y_pred = y_pred.cpu().numpy()
            y_true = y_true.cpu().numpy()

            # Inverse transform to real prices
            y_pred_price = inverse_transform(y_pred.flatten(), scaler, asset_idx, config["d_input"])
            y_true_price = inverse_transform(y_true.flatten(), scaler, asset_idx, config["d_input"])

            # Create plots
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            # Normalised scale
            axes[0].plot(y_true.flatten(), label="Normalised True")
            axes[0].plot(y_pred.flatten(), label="Normalised Predicted")
            axes[0].set_title(f"Batch {i+1} – Normalised")
            axes[0].set_xlabel("Prediction Step")
            axes[0].set_ylabel("Scaled Value")
            axes[0].legend()

            # Real-price scale
            axes[1].plot(y_true_price, label="Real True")
            axes[1].plot(y_pred_price, label="Real Predicted")
            axes[1].set_title(f"Batch {i+1} – Real Prices")
            axes[1].set_xlabel("Prediction Step")
            axes[1].set_ylabel("Price")
            axes[1].legend()

            plt.tight_layout()
            
            # Save or show the plot
            if save_dir:
                plt.savefig(os.path.join(save_dir, f'validation_batch_{i+1}.png'))
                plt.close()
            else:
                plt.show()

            # Lead–lag metrics on real prices
            try:
                ll_metrics = lead_lag_grid(
                    y_true_prices=y_true_price,
                    y_pred_prices=y_pred_price,
                    max_lag=int(config.get("leadlag_max_lag", 24)),
                    return_kind=config.get("leadlag_return_kind", "diff"),
                    min_abs_return=config.get("leadlag_min_abs_return", None),
                )
                ll_fig = plot_lead_lag(ll_metrics, use_plotly=False)
                if save_dir:
                    ll_fig.savefig(os.path.join(save_dir, f'leadlag_batch_{i+1}.png'), bbox_inches='tight')
                    plt.close(ll_fig)
                    with open(os.path.join(save_dir, f'leadlag_batch_{i+1}.txt'), 'w') as f:
                        f.write(lead_lag_report(ll_metrics))
                _ll_summary = ll_metrics.get("summary", {})
            except Exception as e:
                _ll_summary = {}

            # Calculate metrics
            true = y_true.flatten()
            pred = y_pred.flatten()

            # --- Jaggedness metrics ---
            def mean_abs_diff(x):
                return np.mean(np.abs(np.diff(x)))
            
            pred_jagg = mean_abs_diff(pred)
            true_jagg = mean_abs_diff(true)
            jagg_ratio = pred_jagg / (true_jagg + 1e-8)  # avoid divide by zero
            
            metrics.append({
                'batch': i+1,
                'true_std': float(np.std(true)),
                'pred_std': float(np.std(pred)),
                'mse': float(np.mean((true - pred) ** 2)),
                'mae': float(np.mean(np.abs(true - pred))),
                'true_jaggedness': float(true_jagg),
                'pred_jaggedness': float(pred_jagg),
                'jaggedness_ratio': float(jagg_ratio),
                'll_best_lag_by_hit': int(_ll_summary.get('best_lag_by_hit')) if _ll_summary.get('best_lag_by_hit') is not None else None,
                'll_best_hit_ratio': float(_ll_summary.get('best_hit_ratio')) if 'best_hit_ratio' in _ll_summary else np.nan,
                'll_best_lag_by_corr': int(_ll_summary.get('best_lag_by_corr')) if _ll_summary.get('best_lag_by_corr') is not None else None,
                'll_best_corr': float(_ll_summary.get('best_corr')) if 'best_corr' in _ll_summary else np.nan,
                'll_zero_lag_hit': float(_ll_summary.get('zero_lag_hit')) if 'zero_lag_hit' in _ll_summary else np.nan,
                'll_zero_lag_corr': float(_ll_summary.get('zero_lag_corr')) if 'zero_lag_corr' in _ll_summary else np.nan
            })
            
    return metrics

def save_experiment_results(config, train_hist, val_hist, steps_hist, model, val_loader_1, asset_idx, scaler, num_batches=10):
    """Save experimental results including losses, plots, and metrics"""
    # Create experiment directory with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = f"experiments_{timestamp}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Calculate and save total learnable parameters
    total_learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Save configuration with model size
    config_with_params = config.copy()
    config_with_params['total_learnable_parameters'] = total_learnable_params
    with open(os.path.join(exp_dir, 'config.json'), 'w') as f:
        json.dump(config_with_params, f, indent=4)
    
    # Create and save loss plot and data
    fig, df_loss = create_loss_plot(train_hist, val_hist, steps_hist, total_learnable_params)
    fig.write_html(os.path.join(exp_dir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_dir, 'loss_history.csv'))
    
    # Create validation plots and get metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=num_batches,
        save_dir=exp_dir
    )
    
    # Save metrics
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_dir, 'validation_metrics.csv'), index=False)

    # Save dedicated lead-lag summaries per batch
    leadlag_cols = [
        'batch',
        'll_best_lag_by_hit','ll_best_hit_ratio',
        'll_best_lag_by_corr','ll_best_corr',
        'll_zero_lag_hit','ll_zero_lag_corr'
    ]
    available_cols = [c for c in leadlag_cols if c in metrics_df.columns]
    if available_cols:
        leadlag_df = metrics_df[available_cols].copy()
        leadlag_df.to_csv(os.path.join(exp_dir, 'leadlag_summary.csv'), index=False)

        # Aggregate lead-lag overview across batches
        import numpy as _np
        agg = {
            'count_batches': int(len(leadlag_df)),
            'zero_lag_hit_mean': float(_np.nanmean(leadlag_df.get('ll_zero_lag_hit', _np.nan))),
            'zero_lag_corr_mean': float(_np.nanmean(leadlag_df.get('ll_zero_lag_corr', _np.nan))),
            'best_hit_ratio_mean': float(_np.nanmean(leadlag_df.get('ll_best_hit_ratio', _np.nan))),
            'best_corr_mean': float(_np.nanmean(leadlag_df.get('ll_best_corr', _np.nan))),
        }
        # Fractions and means on finite entries only
        blh = leadlag_df.get('ll_best_lag_by_hit')
        blc = leadlag_df.get('ll_best_lag_by_corr')
        if blh is not None:
            vals = _np.array(blh, dtype=float)
            finite_mask = _np.isfinite(vals)
            if finite_mask.any():
                agg['best_lag_by_hit_mean'] = float(_np.nanmean(vals[finite_mask]))
                agg['positive_best_lag_by_hit_frac'] = float(_np.mean(vals[finite_mask] > 0))
        if blc is not None:
            vals = _np.array(blc, dtype=float)
            finite_mask = _np.isfinite(vals)
            if finite_mask.any():
                agg['best_lag_by_corr_mean'] = float(_np.nanmean(vals[finite_mask]))
                agg['positive_best_lag_by_corr_frac'] = float(_np.mean(vals[finite_mask] > 0))

        pd.DataFrame([agg]).to_csv(os.path.join(exp_dir, 'leadlag_aggregate.csv'), index=False)
    
    # Save model summary information
    with open(os.path.join(exp_dir, 'model_summary.txt'), 'w') as f:
        f.write(f"Total Learnable Parameters: {total_learnable_params:,}\n")
        f.write(f"\nModel Configuration:\n")
        for key, value in config_with_params.items():
            f.write(f"{key}: {value}\n")
    
    return exp_dir

In [4]:
# -----------------------------
# Load and preprocess data
# -----------------------------
csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]



In [5]:
# Define parameter grid
param_grid = {
    'd_model': [64],  # Model dimensions
    'distill': [False],   # Whether to use distillation
    'use_time_embedding': [False],  # Whether to use time embeddings
    'dropout': [0.05],  # Dropout rates
    'norm_mode': ['pre','post']  # Normalization mode
}

# Generate all possible combinations
from itertools import product

# Generate all combinations
keys = param_grid.keys()
configs = []
for values in product(*param_grid.values()):
    config_dict = dict(zip(keys, values))
    # Base configuration
    config = {
        "d_input": len(closes.columns),
        "n_heads": 4,
        "enc_layers": 3,
        "dec_layers": 2,
        "enc_len": 96,
        "guiding_len": 48,
        "pred_len": 24,
        "factor": 5,
    }
    # Update with current combination
    config.update(config_dict)
    # Set d_ff to 4x d_model
    config['d_ff'] = config['d_model'] * 4
    configs.append(config)

print(f"Total number of configurations to test: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"\nConfiguration {i+1}:")
    print(f"d_model: {cfg['d_model']}, d_ff: {cfg['d_ff']}")
    print(f"distill: {cfg['distill']}, use_time_embedding: {cfg['use_time_embedding']}")
    print(f"dropout: {cfg['dropout']}")
    print(f"norm_mode: {cfg['norm_mode']}")

Total number of configurations to test: 2

Configuration 1:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.05
norm_mode: pre

Configuration 2:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.05
norm_mode: post


In [6]:
def run_experiment(config, exp_dir, exp_name):
    """Run a single experiment with given configuration"""
    # Set seeds for reproducibility
    set_seed(42)
    
    # Create data loaders
    train_loader, val_loader, scaler, asset_idx = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32,
        val_batch_size=32, 
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Initialize model
    model = InformerForecaster(config, asset_index=asset_idx)
    model.apply(init_weights)
    learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Training configuration
    tcfg = TrainConfig(
        learning_rate=1e-4,
        weight_decay=0.01,
        max_steps=10000,
        warmup_steps=200,
        use_amp=True,
        device="cuda",
        patience=15,
        min_delta=0.0001
    )
    
    # Train model
    model, train_hist, val_hist, steps_hist, best_val_loss, best_val_step = train_model(
        model, train_loader, val_loader, tcfg, asset_index=asset_idx
    )
    
    # Best validation step
    best_step = best_val_step if best_val_step is not None else (min(val_hist, key=lambda t: t[1])[0] if val_hist else float('nan'))
    
    # Create validation loader for visualization
    _, val_loader_1, _, _ = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32, 
        val_batch_size=1,
        val_shuffle=True,
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Create experiment-specific directory
    exp_subdir = os.path.join(exp_dir, exp_name)
    os.makedirs(exp_subdir, exist_ok=True)
    
    # Save model and configuration
    model_save_path = os.path.join(exp_subdir, 'model.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'total_params': learnable_params,
        'train_hist': train_hist,
        'val_hist': val_hist,
        'steps_hist': steps_hist
    }, model_save_path)
    
    # Create and save plots
    fig, df_loss = create_loss_plot(
        train_hist, 
        val_hist, 
        steps_hist,
        total_params=learnable_params
    )
    fig.write_html(os.path.join(exp_subdir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_subdir, 'loss_history.csv'))
    
    # Calculate and save metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=10,
        save_dir=exp_subdir
    )
    
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_subdir, 'metrics.csv'), index=False)
    
    # Save summary metrics
    summary_metrics = metrics_df.mean().round(4)
    
    # Final train and best validation
    final_train_loss = train_hist[-1]
    best_val_loss = best_val_loss if best_val_loss is not None else (min(val_hist, key=lambda t: t[1])[1] if val_hist else float('nan'))
    
    return {
        'Experiment no': exp_name,
        'd_model': config['d_model'],
        'd_ff': config['d_ff'],
        'distill': config['distill'],
        'time embedding': config['use_time_embedding'],
        'dropout': config['dropout'],
        'learnable params': learnable_params,
        'Best Val Step': best_step,
        'train loss': final_train_loss,
        'best validation loss': best_val_loss,
        'jaggedness_ratio (pred/real)': float(summary_metrics['jaggedness_ratio']),
        'MSE': float(summary_metrics['mse']),
        'MAE': float(summary_metrics['mae'])
    }

In [7]:
# Create main experiments directory with timestamp
exp_dir = f"experiments_grid_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(exp_dir, exist_ok=True)

# Run all experiments
results = []
for i, config in enumerate(configs):
    print(f"\nRunning experiment {i+1}/{len(configs)}")
    print("Configuration:", config)
    
    # Create experiment name from parameters
    exp_name = (f"d{config['d_model']}_"
               f"{'dist' if config['distill'] else 'nodist'}_"
               f"{'time' if config['use_time_embedding'] else 'notime'}_"
               f"drop{config['dropout']}"f"{config['norm_mode']}norm")
    
    # Run experiment
    try:
        result = run_experiment(config, exp_dir, exp_name)
        results.append(result)
        print(f"Experiment {exp_name} completed successfully")
        print(f"Best val loss: {result['best validation loss']:.6f}")
        print(f"MSE: {result['MSE']:.6f}")
    except Exception as e:
        print(f"Experiment {exp_name} failed with error: {str(e)}")
        continue

# Create results summary with specific column order
columns = [
    'Experiment no', 
    # Model Params
    'd_model', 'd_ff', 'distill', 'time embedding', 'dropout', 'learnable params',
    # Results
    'Best Val Step', 'train loss', 'best validation loss', 
    'jaggedness_ratio (pred/real)', 'MSE', 'MAE'
]

results_df = pd.DataFrame(results)[columns]

# Save results with proper formatting
results_df.to_csv(os.path.join(exp_dir, 'all_results.csv'), index=False, float_format='%.6f')

# Create summary visualizations
fig = px.scatter(results_df, 
                 x='best validation loss', 
                 y='MSE',
                 hover_data=columns,
                 title='Validation Loss vs MSE across experiments',
                 labels={'Experiment no': 'Experiment Name'})  # Update label
fig.write_html(os.path.join(exp_dir, 'results_scatter.html'))

# Save experiment configuration summary
config_summary = {
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'total_experiments': len(configs),
    'parameter_grid': param_grid,
    'base_config': {k: v for k, v in configs[0].items() if k not in param_grid},
    'experiment_names': [r['Experiment no'] for r in results]
}
with open(os.path.join(exp_dir, 'experiment_config.json'), 'w') as f:
    json.dump(config_summary, f, indent=4)

# Print best models by different metrics
print("\nBest models by validation loss:")
print(results_df.nsmallest(3, 'best validation loss')[columns])

print("\nBest models by MSE:")
print(results_df.nsmallest(3, 'MSE')[columns])

2025-10-29 15:21:57,996 | INFO | num decayed parameter tensors: 41, with 279,872 parameters
2025-10-29 15:21:57,996 | INFO | num non-decayed parameter tensors: 69, with 5,313 parameters
2025-10-29 15:21:57,997 | INFO | Using fused AdamW: True



Running experiment 1/2
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': False, 'dropout': 0.05, 'norm_mode': 'pre', 'd_ff': 256}


2025-10-29 15:21:59,198 | INFO | [Step     0] train_loss=1.054719 | lr=0.000e+00 | samples/s=90.8
2025-10-29 15:22:00,474 | INFO | [Step    10] train_loss=0.919928 | lr=5.000e-06 | samples/s=25.1
2025-10-29 15:22:00,755 | INFO | [Step    20] train_loss=1.186586 | lr=1.000e-05 | samples/s=114.2
2025-10-29 15:22:01,087 | INFO | [Step    30] train_loss=1.197551 | lr=1.500e-05 | samples/s=97.0
2025-10-29 15:22:01,454 | INFO | [Step    40] train_loss=1.189054 | lr=2.000e-05 | samples/s=87.2
2025-10-29 15:22:01,741 | INFO | [Step    50] train_loss=0.841140 | lr=2.500e-05 | samples/s=111.9
2025-10-29 15:22:02,017 | INFO | [Step    60] train_loss=1.115494 | lr=3.000e-05 | samples/s=116.8
2025-10-29 15:22:02,281 | INFO | [Step    70] train_loss=1.087815 | lr=3.500e-05 | samples/s=121.2
2025-10-29 15:22:02,549 | INFO | [Step    80] train_loss=0.967189 | lr=4.000e-05 | samples/s=119.8
2025-10-29 15:22:02,828 | INFO | [Step    90] train_loss=0.842361 | lr=4.500e-05 | samples/s=114.7
2025-10-29 15:

Experiment d64_nodist_notime_drop0.05prenorm completed successfully
Best val loss: 0.004764
MSE: 0.008700

Running experiment 2/2
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': False, 'dropout': 0.05, 'norm_mode': 'post', 'd_ff': 256}


2025-10-29 15:24:57,195 | INFO | [Step    10] train_loss=0.921082 | lr=5.000e-06 | samples/s=24.4
2025-10-29 15:24:57,457 | INFO | [Step    20] train_loss=1.187161 | lr=1.000e-05 | samples/s=123.1
2025-10-29 15:24:57,724 | INFO | [Step    30] train_loss=1.196183 | lr=1.500e-05 | samples/s=120.3
2025-10-29 15:24:57,973 | INFO | [Step    40] train_loss=1.192987 | lr=2.000e-05 | samples/s=129.6
2025-10-29 15:24:58,263 | INFO | [Step    50] train_loss=0.845427 | lr=2.500e-05 | samples/s=110.9
2025-10-29 15:24:58,543 | INFO | [Step    60] train_loss=1.126377 | lr=3.000e-05 | samples/s=114.7
2025-10-29 15:24:58,821 | INFO | [Step    70] train_loss=1.100504 | lr=3.500e-05 | samples/s=115.9
2025-10-29 15:24:59,092 | INFO | [Step    80] train_loss=0.982913 | lr=4.000e-05 | samples/s=119.0
2025-10-29 15:24:59,390 | INFO | [Step    90] train_loss=0.876527 | lr=4.500e-05 | samples/s=108.1
2025-10-29 15:24:59,671 | INFO | [Step   100] train_loss=0.599465 | lr=5.000e-05 | samples/s=114.7
2025-10-29 

Experiment d64_nodist_notime_drop0.05postnorm completed successfully
Best val loss: 0.002431
MSE: 0.001200

Best models by validation loss:
                        Experiment no  d_model  d_ff  distill  time embedding  \
1  d64_nodist_notime_drop0.05postnorm       64   256    False           False   
0   d64_nodist_notime_drop0.05prenorm       64   256    False           False   

   dropout  learnable params  Best Val Step  train loss  best validation loss  \
1     0.05            284929           4100    0.001828              0.002431   
0     0.05            285185           3500    0.002495              0.004764   

   jaggedness_ratio (pred/real)     MSE     MAE  
1                        0.8056  0.0012  0.0275  
0                        1.1008  0.0087  0.0755  

Best models by MSE:
                        Experiment no  d_model  d_ff  distill  time embedding  \
1  d64_nodist_notime_drop0.05postnorm       64   256    False           False   
0   d64_nodist_notime_drop0.05prenorm  